# Método guloso: códigos de Huffman e o argumento de troca — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula. O código dos algoritmos é escrito em C e compilado com `gcc`
a partir do próprio notebook: cada célula `%%writefile` grava um programa em `src/`, e a célula
seguinte compila e executa. As verificações, os gráficos e alguns exercícios ficam em Python.

## Objetivos

Ao final deste tutorial você será capaz de:

- implementar a **seleção de intervalos** pelo critério "termina mais cedo" e testar critérios gulosos contra a força bruta;
- implementar a **mochila fracionária** pelo critério "maior valor por quilo", que falha na mochila 0/1;
- verificar se um código é **de prefixo**, calcular seu custo $B(T)$ e **decodificar** uma sequência de bits;
- implementar o algoritmo de **Huffman** com um heap mínimo em $O(n \log n)$;
- conferir experimentalmente a **otimalidade** de Huffman contra a força bruta e contra um método guloso que falha (Shannon–Fano).

In [ ]:
# Preparação do ambiente: pasta de trabalho, compilador e funções auxiliares.
import os, subprocess, math, random, itertools

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

_compilados = {}   # fonte -> data de modificação da última compilação bem-sucedida

def compilar(fonte):
    """Compila src/<fonte> com gcc -O2 (só se o arquivo mudou); devolve o executável ou None."""
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    mtime = os.path.getmtime(os.path.join('src', fonte))
    if _compilados.get(fonte) == mtime and os.path.exists(exe):
        return exe
    comp = subprocess.run(['gcc', '-std=c99', '-Wall', '-O2', '-o', exe,
                           os.path.join('src', fonte), '-lm'],
                          capture_output=True, text=True)
    if comp.returncode != 0:
        print('ERRO DE COMPILAÇÃO:\n' + comp.stderr)
        return None
    if comp.stderr:
        print(comp.stderr)
    _compilados[fonte] = mtime
    return exe

def rodar(fonte, args=(), entrada=None, mostrar=True):
    """Compila src/<fonte>, executa com os argumentos e a entrada dados e devolve a saída."""
    exe = compilar(fonte)
    if exe is None:
        return ''
    r = subprocess.run([exe, *map(str, args)], input=entrada, capture_output=True, text=True,
                       errors='replace')
    if mostrar:
        print(r.stdout, end='')
    if r.returncode != 0:
        print(f'(programa terminou com código {r.returncode})', r.stderr)
    return r.stdout

def campo(saida, nome):
    """Extrai o valor de 'nome=valor' da saída (int se for inteiro, float se for número, senão str)."""
    for tok in saida.split():
        if tok.startswith(nome + '='):
            valor = tok.split('=', 1)[1]
            if valor.lstrip('-').isdigit():
                return int(valor)
            try:
                return float(valor)
            except ValueError:
                return valor
    raise ValueError(f'{nome} não encontrado em: {saida!r}')

def linha(saida, rotulo):
    """Devolve os inteiros da linha que começa com 'rotulo:'."""
    for lin in saida.splitlines():
        if lin.startswith(rotulo + ':'):
            return [int(x) for x in lin.split(':', 1)[1].split()]
    raise ValueError(f'linha {rotulo} não encontrada em: {saida!r}')

# Os seis intervalos [s, f) da aula de PD (13/10), agora sem pesos.
INTERVALOS = [(0, 4), (1, 6), (5, 7), (2, 9), (8, 10), (7, 11)]
# Exemplo condutor de Huffman: Cormen et al. (2009), §16.3, Figura 16.3 (frequências em milhares).
FREQ = {'a': 45, 'b': 13, 'c': 12, 'd': 16, 'e': 9, 'f': 5}

## 1. Seleção de intervalos

Dados $n$ intervalos semiabertos $[s_i, f_i)$, queremos o maior subconjunto de intervalos
**dois a dois compatíveis** ($f_i \le s_j$ ou $f_j \le s_i$). O critério guloso correto ordena
por **término** e escolhe cada intervalo que começa depois do último escolhido terminar. O
argumento de troca da aula mostra que existe uma solução ótima que contém o intervalo que
termina primeiro; o custo é dominado pela ordenação, $O(n \log n)$.

In [ ]:
%%writefile src/intervalos.c
/* SELECAO DE INTERVALOS pelo criterio "termina mais cedo".
   Uso: ./intervalos < entrada    (n e depois n linhas "s f"; intervalos [s, f))
   Imprime os intervalos escolhidos (numerados de 1 a n na ordem da entrada) e quantos sao.
   Cormen et al. (2009), sec. 16.1; Kleinberg e Tardos (2006), sec. 4.1. */
#include <stdio.h>
#include <stdlib.h>
#include <limits.h>

typedef struct { int s, f, id; } Intervalo;

/* ordem crescente de termino; empate pelo numero do intervalo */
static int cmp_fim( const void *a, const void *b) {
   const Intervalo *x = a, *y = b;
   if (x->f != y->f) return x->f < y->f ? -1 : 1;
   return x->id - y->id;
}

/* ordena v[0..n-1] por termino e grava em esc[] as posicoes escolhidas; devolve quantas */
int seleciona( int n, Intervalo v[], int esc[]) {
   qsort( v, n, sizeof (Intervalo), cmp_fim);
   int k = 0, fim = INT_MIN;          /* fim = termino do ultimo escolhido */
   for (int i = 0; i < n; ++i)
      if (v[i].s >= fim) {             /* compativel com todos os escolhidos */
         esc[k++] = i;
         fim = v[i].f;
      }
   return k;
}

int main( void) {
   int n;
   if (scanf( "%d", &n) != 1) return 1;
   Intervalo *v = malloc( n * sizeof (Intervalo));
   int *esc = malloc( n * sizeof (int));
   for (int i = 0; i < n; ++i) {
      if (scanf( "%d %d", &v[i].s, &v[i].f) != 2) return 1;
      v[i].id = i + 1;
   }
   int k = seleciona( n, v, esc);
   printf( "escolhidos:");
   for (int t = 0; t < k; ++t) printf( " %d", v[esc[t]].id);
   printf( "\nk=%d\n", k);
   free( v);  free( esc);
   return 0;
}

In [ ]:
# Exemplo: o programa em C na instância dos slides.
def entrada_intervalos(intervalos):
    return f'{len(intervalos)}\n' + ''.join(f'{s} {f}\n' for s, f in intervalos)

saida = rodar('intervalos.c', entrada=entrada_intervalos(INTERVALOS))
assert linha(saida, 'escolhidos') == [1, 3, 5], 'Nos slides, o guloso escolhe 1, 3 e 5'

Nos slides, três outros critérios naturais caem com contraexemplos. A célula abaixo implementa
os quatro critérios de forma genérica (a cada passo, escolher o intervalo restante que minimiza
o critério e descartar os incompatíveis) e compara cada um com a **força bruta**, que testa
todos os subconjuntos.

In [ ]:
# Exemplo: os quatro critérios gulosos contra a força bruta, nos contraexemplos dos slides.
def compativeis(a, b):
    return a[1] <= b[0] or b[1] <= a[0]

def guloso_por(intervalos, criterio):
    """Escolhe repetidamente o intervalo restante de menor criterio(i, restantes)."""
    restantes, escolhidos = list(range(len(intervalos))), []
    while restantes:
        i = min(restantes, key=lambda i: (criterio(intervalos, i, restantes), i))
        escolhidos.append(i)
        restantes = [j for j in restantes
                     if j != i and compativeis(intervalos[i], intervalos[j])]
    return escolhidos

CRITERIOS = {
    'começa mais cedo': lambda I, i, R: I[i][0],
    'mais curto':       lambda I, i, R: I[i][1] - I[i][0],
    'menos conflitos':  lambda I, i, R: sum(not compativeis(I[i], I[j]) for j in R if j != i),
    'termina mais cedo': lambda I, i, R: I[i][1],
}

def otimo_forca_bruta(intervalos):
    for r in range(len(intervalos), 0, -1):
        for sub in itertools.combinations(intervalos, r):
            if all(compativeis(a, b) for a, b in itertools.combinations(sub, 2)):
                return r
    return 0

CONTRAEXEMPLOS = {
    'começa mais cedo': [(0, 12), (1, 3), (4, 6), (7, 9), (10, 12)],
    'mais curto':       [(0, 6), (6, 12), (5, 7)],
    'menos conflitos':  [(0, 3), (3, 6), (6, 9), (9, 12), (5, 7),
                         (2, 4), (2, 4), (2, 4), (8, 10), (8, 10), (8, 10)],
    'slides (13/10)':   INTERVALOS,
}

print(f'{"instância":18s}' + ''.join(f'{c:>19s}' for c in CRITERIOS) + '   ótimo')
for nome, I in CONTRAEXEMPLOS.items():
    print(f'{nome:18s}' + ''.join(f'{len(guloso_por(I, c)):19d}' for c in CRITERIOS.values())
          + f'{otimo_forca_bruta(I):8d}')

In [ ]:
# Visualização: a escolha gulosa na instância dos slides (laranja = escolhido).
import matplotlib.pyplot as plt

escolhidos = guloso_por(INTERVALOS, CRITERIOS['termina mais cedo'])
fig, ax = plt.subplots(figsize=(7, 2.8))
for i, (s, f) in enumerate(INTERVALOS):
    cor = 'tab:orange' if i in escolhidos else 'lightgray'
    ax.plot([s, f], [i + 1, i + 1], color=cor, linewidth=7, solid_capstyle='butt')
for i in escolhidos:
    ax.axvline(INTERVALOS[i][1], color='tab:orange', linestyle=':', linewidth=1)
ax.set_yticks(range(1, len(INTERVALOS) + 1))
ax.set_xticks(range(0, 12))
ax.invert_yaxis()
ax.set_xlabel('tempo'); ax.set_ylabel('intervalo')
ax.set_title('Termina mais cedo: escolhe 1, 3 e 5')
plt.show()

### Exercício 1: mochila fracionária

Na mochila **fracionária** podemos levar qualquer fração $x_i \in [0, 1]$ de cada item, e o
critério guloso "maior valor por quilo" ($v_i / w_i$) é ótimo. Complete
`mochila_fracionaria()`: ordene os itens por $v_i/w_i$ decrescente (escreva o comparador do
`qsort`, de preferência comparando $v_x w_y$ com $v_y w_x$ para evitar divisões) e encha a
mochila com itens inteiros enquanto couberem; o próximo item entra com a fração que cabe.

O teste confere a instância dos slides (Cormen, Figura 16.2: $W = 50$, resposta 240) e 300
instâncias aleatórias contra um cálculo independente.

In [ ]:
%%writefile src/mochila_frac.c
/* Exercicio 1: MOCHILA FRACIONARIA pelo criterio guloso "maior valor por quilo".
   Uso: ./mochila_frac < entrada   (instancias "n W" seguidas de n linhas "w v"; termina em "0 0")
   Para cada instancia imprime "val=<valor otimo>".
   TODO: complete mochila_fracionaria(). */
#include <stdio.h>
#include <stdlib.h>

typedef struct { int w, v; } Item;

double mochila_fracionaria( int n, Item it[], int W) {
   /* TODO: ordene it[] por v/w decrescente (qsort com um comparador seu) e encha a mochila:
      itens inteiros enquanto couberem, depois uma fracao do proximo. */
   return 0.0;
}

int main( void) {
   int n, W;
   while (scanf( "%d %d", &n, &W) == 2 && n > 0) {
      Item *it = malloc( n * sizeof (Item));
      for (int i = 0; i < n; ++i)
         if (scanf( "%d %d", &it[i].w, &it[i].v) != 2) return 1;
      printf( "val=%.6f\n", mochila_fracionaria( n, it, W));
      free( it);
   }
   return 0;
}

In [ ]:
# Teste automático do Exercício 1
def referencia_frac(itens, W):
    """Valor ótimo pela dualidade de programação linear (não usa o critério guloso):
    min sobre lam >= 0 de lam*W + soma de max(0, v - lam*w); o mínimo ocorre em lam = 0
    ou em algum lam = v_i / w_i."""
    candidatos = [0.0] + [v / w for w, v in itens]
    return min(lam * W + sum(max(0.0, v - lam * w) for w, v in itens) for lam in candidatos)

random.seed(2026)
instancias = [([(10, 60), (20, 100), (30, 120)], 50)]
for _ in range(300):
    n = random.randint(1, 30)
    itens = [(random.randint(1, 40), random.randint(1, 100)) for _ in range(n)]
    instancias.append((itens, random.randint(1, 300)))
entrada = ''.join(f'{len(it)} {W}\n' + ''.join(f'{w} {v}\n' for w, v in it)
                  for it, W in instancias) + '0 0\n'
saida = rodar('mochila_frac.c', entrada=entrada, mostrar=False)
vals = [float(t.split('=')[1]) for t in saida.split() if t.startswith('val=')]
assert len(vals) == len(instancias), 'O programa deve imprimir um val= por instância'
assert abs(vals[0] - 240) < 1e-6, f'Cormen, Figura 16.2: esperado 240, obtido {vals[0]}'
for k, ((itens, W), val) in enumerate(zip(instancias, vals)):
    ref = referencia_frac(itens, W)
    assert abs(val - ref) < 1e-4, f'Instância {k}: obtido {val}, esperado {ref:.6f}'
print('Exercício 1 ok!')

## 2. Códigos de prefixo

Um **código de prefixo** atribui a cada caractere uma palavra binária de modo que nenhuma
palavra é prefixo de outra; por isso a decodificação nunca é ambígua. Para frequências
$c.\mathit{freq}$ e palavras de tamanho $d(c)$, o custo de codificar o texto é
$B = \sum_c c.\mathit{freq} \cdot d(c)$ bits. No exemplo condutor (Cormen, §16.3), o código
de tamanho fixo gasta 300 mil bits e o de tamanho variável, 224 mil.

In [ ]:
# Exemplo: verificar a propriedade de prefixo e calcular o custo B.
FIXO     = {'a': '000', 'b': '001', 'c': '010', 'd': '011', 'e': '100', 'f': '101'}
VARIAVEL = {'a': '0', 'b': '101', 'c': '100', 'd': '111', 'e': '1101', 'f': '1100'}
AMBIGUO  = {'a': '0', 'b': '01', 'c': '1'}            # "01" pode ser "b" ou "ac"

def eh_prefixo(codigo):
    """Em ordem lexicográfica, se uma palavra é prefixo de outra, também é prefixo da
    palavra seguinte; basta comparar vizinhas."""
    ws = sorted(codigo.values())
    return all(not ws[i + 1].startswith(ws[i]) for i in range(len(ws) - 1))

def custo(codigo, freq):
    return sum(freq[c] * len(codigo[c]) for c in freq)

for nome, cod in [('fixo', FIXO), ('variável', VARIAVEL), ('ambíguo', AMBIGUO)]:
    print(f'{nome:9s} prefixo={eh_prefixo(cod)!s:5s}',
          f'B={custo(cod, FREQ)} mil bits' if set(cod) == set(FREQ) else '')

### Exercício 2: decodificação

Complete `decodifica(bits, codigo)`, que recebe uma cadeia de `'0'`/`'1'` e um código de
prefixo (dicionário caractere → palavra) e devolve o texto original. Leia os bits da esquerda
para a direita acumulando um pedaço; quando o pedaço for uma palavra do código, emita o
caractere e recomece. Se sobrarem bits que não formam palavra, levante `ValueError`.

In [ ]:
# Exercício 2: decodificação de um código de prefixo
# TODO: implemente decodifica(bits, codigo).

def decodifica(bits, codigo):
    # TODO: implemente aqui
    pass


# Teste automático
assert decodifica('010111001101', VARIAVEL) == 'abfe', 'Slides: 0 101 1100 1101 = abfe'
assert decodifica('', VARIAVEL) == '', 'Sequência vazia decodifica para texto vazio'
assert decodifica('000001010011100101', FIXO) == 'abcdef'
texto = 'cafe' * 5 + 'abadedafeca'
assert decodifica(''.join(VARIAVEL[c] for c in texto), VARIAVEL) == texto
try:
    decodifica('11', VARIAVEL)
    sobrou_sem_erro = True
except ValueError:
    sobrou_sem_erro = False
assert not sobrou_sem_erro, 'Bits que sobram sem formar palavra devem levantar ValueError'
print('Exercício 2 ok!')

## 3. O algoritmo de Huffman

Huffman começa com $n$ árvores de um nó e repete $n-1$ vezes: tira do heap as duas árvores de
**menor** frequência $x$ e $y$, cria um nó $z$ com filhos $x$ e $y$ e
$z.\mathit{freq} = x.\mathit{freq} + y.\mathit{freq}$, e devolve $z$ ao heap. Cada iteração faz
três operações de heap, logo o total é $O(n \log n)$. Como nos slides, a árvore fica em vetores:
os nós $0..n-1$ são as folhas e $n..2n-2$ os nós internos, com filhos em `esq[]` e `dir[]`.

In [ ]:
%%writefile src/huffman.c
/* Algoritmo de HUFFMAN com heap minimo.
   Uso: ./huffman < entrada    (n e depois n linhas "c f": um caractere sem espacos e a frequencia)
   Imprime os nos internos ("no: z x y freq"), as palavras ("cod_c=...") e o custo B(T).
   Huffman (1952); Cormen et al. (2009), sec. 16.3; Levitin (2012), sec. 9.4. */
#include <stdio.h>

#define MAXN 256                          /* maximo de caracteres */

int freq[2*MAXN], esq[2*MAXN], dir[2*MAXN];   /* nos 0..n-1 folhas, n..2n-2 internos */

/* heap minimo de nos, ordenado por freq (empate: menor indice primeiro) */
static int heap[2*MAXN], tam = 0;

static int menor( int a, int b) {
   return freq[a] < freq[b] || (freq[a] == freq[b] && a < b);
}

void heap_insere( int x) {
   int i = tam++;
   while (i > 0 && menor( x, heap[(i-1)/2])) {   /* sobe enquanto for menor que o pai */
      heap[i] = heap[(i-1)/2];
      i = (i-1)/2;
   }
   heap[i] = x;
}

int heap_remove_min( void) {
   int min = heap[0], x = heap[--tam], i = 0;
   for (;;) {                                    /* desce x a partir da raiz */
      int f = 2*i + 1;
      if (f >= tam) break;
      if (f + 1 < tam && menor( heap[f+1], heap[f])) ++f;
      if (!menor( heap[f], x)) break;
      heap[i] = heap[f];
      i = f;
   }
   heap[i] = x;
   return min;
}

/* constroi a arvore de Huffman das folhas 0..n-1 e devolve a raiz */
int huffman( int n) {
   tam = 0;
   for (int i = 0; i < n; ++i)
      heap_insere( i);
   for (int z = n; z < 2*n - 1; ++z) {           /* n-1 fusoes */
      int x = heap_remove_min();                 /* os dois de menor frequencia */
      int y = heap_remove_min();
      esq[z] = x;  dir[z] = y;
      freq[z] = freq[x] + freq[y];
      heap_insere( z);
   }
   return heap_remove_min();                     /* a raiz, 2n-2 */
}

static char nome[MAXN], palavra[MAXN];

/* DFS da raiz: 0 ao descer a esquerda, 1 a direita; imprime a palavra de cada folha */
static void codigos( int v, int prof, int n) {
   if (v < n) {
      palavra[prof] = '\0';
      printf( "cod_%c=%s\n", nome[v], prof > 0 ? palavra : "0");
      return;
   }
   palavra[prof] = '0';  codigos( esq[v], prof + 1, n);
   palavra[prof] = '1';  codigos( dir[v], prof + 1, n);
}

int main( void) {
   int n;
   if (scanf( "%d", &n) != 1 || n < 1 || n > MAXN) return 1;
   for (int i = 0; i < n; ++i)
      if (scanf( " %c %d", &nome[i], &freq[i]) != 2) return 1;
   int raiz = huffman( n);
   long B = 0;
   for (int z = n; z < 2*n - 1; ++z) {           /* B(T) = soma dos nos internos */
      printf( "no: %d %d %d %d\n", z, esq[z], dir[z], freq[z]);
      B += freq[z];
   }
   codigos( raiz, 0, n);
   printf( "B=%ld\n", B);
   return 0;
}

In [ ]:
# Exemplo: Huffman no exemplo condutor (Cormen, Figura 16.5).
def entrada_freq(freq):
    return f'{len(freq)}\n' + ''.join(f'{c} {f}\n' for c, f in freq.items())

def palavras(saida):
    return {t[4]: t.split('=', 1)[1] for t in saida.split() if t.startswith('cod_')}

saida = rodar('huffman.c', entrada=entrada_freq(FREQ))
assert palavras(saida) == VARIAVEL, 'Deve reproduzir o código variável dos slides'
assert campo(saida, 'B') == 224

In [ ]:
# Visualização: a árvore de Huffman desenhada a partir da saída do programa em C.
def desenha_arvore(saida, nomes, titulo):
    nos = {}
    for lin in saida.splitlines():
        if lin.startswith('no:'):
            z, x, y, f = map(int, lin.split()[1:])
            nos[z] = (x, y, f)
    n, raiz = len(nomes), max(nos)
    freqs = list(FREQ.values()) if nomes == list(FREQ) else None
    pos, prox_x = {}, [0]
    def layout(v, prof):                       # folhas da esquerda para a direita
        if v < n:
            pos[v] = (prox_x[0], -prof); prox_x[0] += 1
        else:
            x, y, _ = nos[v]
            layout(x, prof + 1); layout(y, prof + 1)
            pos[v] = ((pos[x][0] + pos[y][0]) / 2, -prof)
    layout(raiz, 0)
    fig, ax = plt.subplots(figsize=(7, 3.6))
    for z, (x, y, f) in nos.items():
        for filho, bit in ((x, '0'), (y, '1')):
            ax.plot(*zip(pos[z], pos[filho]), color='gray', linewidth=1, zorder=1)
            mx, my = (pos[z][0] + pos[filho][0]) / 2, (pos[z][1] + pos[filho][1]) / 2
            ax.text(mx, my, bit, fontsize=8, color='dimgray', ha='center', va='bottom')
        ax.scatter(*pos[z], s=500, color='navajowhite', edgecolor='darkorange', zorder=2)
        ax.text(*pos[z], str(f), ha='center', va='center', fontsize=8, zorder=3)
    for v in range(n):
        rot = nomes[v] + (f':{freqs[v]}' if freqs else '')
        ax.text(*pos[v], rot, ha='center', va='center', fontsize=8,
                bbox=dict(boxstyle='round', fc='lightsteelblue', ec='steelblue'))
    ax.set_title(titulo); ax.axis('off')
    plt.show()

desenha_arvore(saida, list(FREQ), 'Árvore de Huffman do exemplo condutor: B(T) = 224')

### Exercício 3: Huffman com heap

O programa abaixo já tem o heap mínimo (`heap_insere` e `heap_remove_min`, que comparam os
nós por `freq[]`). Complete `huffman(n)`: ele recebe as folhas `0..n-1` com `freq[]`
preenchido, deve criar os nós internos `n..2n-2` preenchendo `esq[]`, `dir[]` e `freq[]`, e
devolver a raiz. Não use o código da Seção 3 — escreva o laço você mesmo.

O `main` confere o exemplo condutor e 300 instâncias aleatórias: verifica que a árvore é
válida (cada folha aparece uma vez) e que o custo é o mínimo, calculado por outro método.

In [ ]:
%%writefile src/huffman_ex3.c
/* Exercicio 3: algoritmo de HUFFMAN.
   TODO: complete huffman(). O main confere a arvore com outro calculo do custo otimo. */
#include <stdio.h>
#include <stdlib.h>

#define MAXN 256                          /* maximo de caracteres */

int freq[2*MAXN], esq[2*MAXN], dir[2*MAXN];   /* nos 0..n-1 folhas, n..2n-2 internos */

/* heap minimo de nos, ordenado por freq (empate: menor indice primeiro) */
static int heap[2*MAXN], tam = 0;

static int menor( int a, int b) {
   return freq[a] < freq[b] || (freq[a] == freq[b] && a < b);
}

void heap_insere( int x) {
   int i = tam++;
   while (i > 0 && menor( x, heap[(i-1)/2])) {   /* sobe enquanto for menor que o pai */
      heap[i] = heap[(i-1)/2];
      i = (i-1)/2;
   }
   heap[i] = x;
}

int heap_remove_min( void) {
   int min = heap[0], x = heap[--tam], i = 0;
   for (;;) {                                    /* desce x a partir da raiz */
      int f = 2*i + 1;
      if (f >= tam) break;
      if (f + 1 < tam && menor( heap[f+1], heap[f])) ++f;
      if (!menor( heap[f], x)) break;
      heap[i] = heap[f];
      i = f;
   }
   heap[i] = x;
   return min;
}

/* folhas 0..n-1 com freq[] dado; crie os nos internos n..2n-2 (esq[], dir[], freq[])
   e devolva a raiz */
int huffman( int n) {
   /* TODO: zere o heap (tam = 0), insira as folhas e faca as n-1 fusoes */
   return -1;
}

/* ---------- verificacao (nao altere) ---------- */
static int vistos[MAXN];

/* soma freq * profundidade das folhas; devolve -1 se a arvore for invalida */
static long custo_arvore( int v, int prof, int n) {
   if (v < 0 || v > 2*n - 2) return -1;
   if (v < n) { vistos[v]++; return (long) freq[v] * prof; }
   long a = custo_arvore( esq[v], prof + 1, n), b = custo_arvore( dir[v], prof + 1, n);
   return (a < 0 || b < 0) ? -1 : a + b;
}

/* custo otimo por outro caminho: em vez do heap, varre um vetor procurando os dois menores */
static long custo_referencia( int n, const int f0[]) {
   long ativo[MAXN], B = 0;
   int m = n;
   for (int i = 0; i < n; ++i) ativo[i] = f0[i];
   while (m > 1) {
      int a = 0;
      for (int i = 1; i < m; ++i) if (ativo[i] < ativo[a]) a = i;
      long fa = ativo[a];  ativo[a] = ativo[--m];
      int b = 0;
      for (int i = 1; i < m; ++i) if (ativo[i] < ativo[b]) b = i;
      ativo[b] += fa;  B += ativo[b];
   }
   return B;
}

static int confere( int n, const int f0[], const char *rotulo) {
   for (int i = 0; i < n; ++i) freq[i] = f0[i];
   for (int i = 0; i < 2*n; ++i) esq[i] = dir[i] = -1;
   int raiz = huffman( n);
   for (int i = 0; i < n; ++i) vistos[i] = 0;
   long B = raiz == 2*n - 2 ? custo_arvore( raiz, 0, n) : -1;
   for (int i = 0; i < n; ++i) if (vistos[i] != 1) B = -1;
   long ref = custo_referencia( n, f0);
   if (B != ref) {
      if (B < 0) printf( "ERRO (%s, n=%d): arvore invalida ou raiz diferente de 2n-2\n", rotulo, n);
      else       printf( "ERRO (%s, n=%d): B=%ld, minimo %ld\n", rotulo, n, B, ref);
      return 0;
   }
   return 1;
}

int main( void) {
   int clrs[] = {45, 13, 12, 16, 9, 5};             /* a b c d e f */
   for (int i = 0; i < 6; ++i) freq[i] = clrs[i];
   int raiz = huffman( 6);
   long B = 0;
   for (int z = 6; z < 11; ++z) B += freq[z];
   printf( "raiz=%d B=%ld\n", raiz, B);
   if (!confere( 6, clrs, "Cormen")) { printf( "ok=0\n"); return 0; }
   srand( 2026);
   for (int t = 0; t < 300; ++t) {
      int n = 2 + rand() % (MAXN - 1), f0[MAXN];
      for (int i = 0; i < n; ++i) f0[i] = 1 + rand() % (t % 2 ? 10 : 100000);
      if (!confere( n, f0, "aleatoria")) { printf( "ok=0\n"); return 0; }
   }
   printf( "ok=1\n");
   return 0;
}

In [ ]:
# Teste automático do Exercício 3
saida = rodar('huffman_ex3.c')
assert campo(saida, 'B') == 224, 'No exemplo condutor, B(T) = 14 + 25 + 30 + 55 + 100 = 224'
assert campo(saida, 'raiz') == 10, 'Com n = 6, a raiz é o nó 2n - 2 = 10'
assert campo(saida, 'ok') == 1, 'Verifique sua implementação (veja o ERRO impresso acima)'
print('Exercício 3 ok!')

In [ ]:
%%writefile src/huffman_tempo.c
/* Mede o tempo de Huffman com n folhas de frequencias aleatorias (vetores dinamicos).
   Uso: ./huffman_tempo n */
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

static long *freq;
static int *heap, tam;

static int menor( int a, int b) { return freq[a] < freq[b] || (freq[a] == freq[b] && a < b); }

static void insere( int x) {
   int i = tam++;
   while (i > 0 && menor( x, heap[(i-1)/2])) { heap[i] = heap[(i-1)/2]; i = (i-1)/2; }
   heap[i] = x;
}

static int remove_min( void) {
   int min = heap[0], x = heap[--tam], i = 0;
   for (;;) {
      int f = 2*i + 1;
      if (f >= tam) break;
      if (f + 1 < tam && menor( heap[f+1], heap[f])) ++f;
      if (!menor( heap[f], x)) break;
      heap[i] = heap[f];  i = f;
   }
   heap[i] = x;
   return min;
}

int main( int argc, char *argv[]) {
   int n = atoi( argv[1]);
   freq = malloc( 2 * n * sizeof (long));
   heap = malloc( 2 * n * sizeof (int));
   srand( 1);
   for (int i = 0; i < n; ++i) freq[i] = 1 + rand() % 1000000;
   clock_t t0 = clock();
   tam = 0;
   for (int i = 0; i < n; ++i) insere( i);
   for (int z = n; z < 2*n - 1; ++z) {
      int x = remove_min(), y = remove_min();
      freq[z] = freq[x] + freq[y];
      insere( z);
   }
   printf( "raiz=%d t=%.6f\n", remove_min(), (double) (clock() - t0) / CLOCKS_PER_SEC);
   return 0;
}

In [ ]:
# Visualização: o tempo de Huffman dividido por n log n fica praticamente constante.
ns = [2**k for k in range(14, 22)]
ts = [campo(rodar('huffman_tempo.c', args=(n,), mostrar=False), 't') for n in ns]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
a1.plot(ns, ts, 'o-')
a1.set_xlabel('n (caracteres)'); a1.set_ylabel('tempo (s)'); a1.set_title('Huffman com heap')
a2.plot(ns, [t / (n * math.log2(n)) * 1e9 for t, n in zip(ts, ns)], 's-', color='tab:orange')
a2.set_xscale('log', base=2); a2.set_ylim(bottom=0)
a2.set_xlabel('n'); a2.set_ylabel('ns por (n log n)'); a2.set_title('tempo / (n log n)')
plt.tight_layout(); plt.show()

## 4. Huffman é ótimo: conferindo contra a força bruta

A prova da aula tem dois lemas: juntar os dois menos frequentes é seguro (argumento de troca)
e o que sobra é um problema de Huffman menor (subestrutura ótima). Aqui conferimos o resultado
experimentalmente. Uma observação ajuda: toda árvore cheia pode ser construída por **alguma**
sequência de fusões de pares, e o custo é a soma dos valores criados. Logo o ótimo é o mínimo,
sobre **todos** os pares $(i, j)$ que se podem juntar primeiro, de
$f_i + f_j + \mathrm{ótimo}(\text{lista com } f_i, f_j \text{ trocados por } f_i + f_j)$.

In [ ]:
# Exemplo: o custo de Huffman em Python, com heapq (B = soma dos nós criados).
import heapq

def custo_huffman(freqs):
    h = list(freqs)
    heapq.heapify(h)
    B = 0
    while len(h) > 1:
        z = heapq.heappop(h) + heapq.heappop(h)
        B += z
        heapq.heappush(h, z)
    return B

print('Exemplo condutor:', custo_huffman(FREQ.values()), 'mil bits')

### Exercício 4: o ótimo por força bruta

Complete `otimo_forca_bruta(freqs)`, que devolve o menor custo $B(T)$ entre **todas** as
árvores cheias, usando a recorrência acima: se há no máximo um elemento o custo é 0; senão,
teste todos os pares $i < j$. Use `functools.lru_cache` sobre uma tupla **ordenada** para não
recalcular listas iguais. O teste compara com Huffman em 300 listas aleatórias de até 7
frequências.

In [ ]:
# Exercício 4: custo ótimo por força bruta
# TODO: implemente otimo_forca_bruta(freqs), testando todas as fusões possíveis.
from functools import lru_cache

def otimo_forca_bruta(freqs):
    # TODO: implemente aqui (sugestão: uma função interna com @lru_cache sobre tuple(sorted(...)))
    pass


# Teste automático
assert otimo_forca_bruta([7]) == 0, 'Um caractere só: nenhuma fusão'
assert otimo_forca_bruta(list(FREQ.values())) == 224, 'Exemplo condutor: 224'
random.seed(7)
for _ in range(300):
    fs = [random.randint(1, 50) for _ in range(random.randint(2, 7))]
    assert otimo_forca_bruta(fs) == custo_huffman(fs), f'Diferença em {fs}'
print('Exercício 4 ok! Huffman coincidiu com a força bruta em 300 instâncias.')

Antes de Huffman, o método de **Shannon–Fano** construía o código **de cima para baixo**:
ordena os caracteres por frequência e divide a lista em duas partes de somas o mais parecidas
possível (uma recebe o bit 0, a outra o bit 1), repetindo em cada parte. Também é uma regra
gulosa razoável — mas não é ótima. A célula abaixo procura instâncias em que ela perde.

In [ ]:
# Exemplo: Shannon–Fano (de cima para baixo) contra Huffman (de baixo para cima).
def custo_shannon_fano(freqs):
    fs = sorted(freqs, reverse=True)
    def custo(fs, prof):
        if len(fs) == 1:
            return fs[0] * prof
        total, acum, corte, melhor = sum(fs), 0, 1, math.inf
        for k in range(1, len(fs)):                  # corte mais equilibrado
            acum += fs[k - 1]
            if abs(total - 2 * acum) < melhor:
                melhor, corte = abs(total - 2 * acum), k
        return custo(fs[:corte], prof + 1) + custo(fs[corte:], prof + 1)
    return custo(fs, 0)

print('Exemplo condutor: Shannon–Fano', custo_shannon_fano(FREQ.values()),
      '| Huffman', custo_huffman(FREQ.values()))
random.seed(1)
perdas = 0
for t in range(2000):
    fs = [random.randint(1, 30) for _ in range(random.randint(3, 8))]
    sf, hf = custo_shannon_fano(fs), custo_huffman(fs)
    assert sf >= hf
    if sf > hf:
        perdas += 1
        if perdas == 1:
            print(f'Primeira derrota encontrada: {sorted(fs, reverse=True)} -> SF {sf}, Huffman {hf}')
print(f'Shannon–Fano foi pior em {perdas} de 2000 instâncias aleatórias.')

## Desafio Final: compactando um texto de verdade

Use o texto da célula abaixo (ou qualquer outro arquivo de texto) e compare quatro medidas, em
**bits por caractere**:

1. **ASCII/UTF-8 de 8 bits** (para simplificar, trate cada caractere como 8 bits);
2. o melhor **código de tamanho fixo**, $\lceil \log_2 k \rceil$ bits para $k$ caracteres distintos;
3. o **código de Huffman** das frequências do texto (você pode adaptar `src/huffman.c` — troque
   o espaço e a quebra de linha por símbolos visíveis, como `_` e `~` — ou escrever a versão em
   Python com `heapq`, guardando as palavras e não só o custo);
4. a **entropia** $H = -\sum_c p_c \log_2 p_c$, o limite inferior de Shannon.

Depois **codifique** o texto inteiro com o seu código de Huffman, **decodifique** com o
`decodifica()` do Exercício 2 e confira que volta igual. Confirme também que
$H \le L < H + 1$, em que $L$ é o comprimento médio do seu código.

*Para ir além:* calcule o mesmo para pares de caracteres (bigramas) como símbolos. Por que os
bits por caractere caem? O que isso diz sobre a hipótese de caracteres independentes?

In [ ]:
# Desafio: compactando um texto com Huffman
# TODO: resolva o desafio abaixo

TEXTO = """O algoritmo de Huffman foi publicado em 1952 por David Huffman, então aluno de
doutorado no MIT. A ideia é simples: os caracteres mais frequentes recebem as palavras mais
curtas, e a árvore é construída de baixo para cima, juntando sempre as duas subárvores de
menor frequência. Setenta anos depois, a mesma construção continua no estágio final de
formatos como DEFLATE, usado em arquivos zip, gzip e PNG, e no JPEG."""

# Sua solução aqui

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. As principais para este tutorial:

- CORMEN, T. H. et al. *Introduction to Algorithms*. 3. ed. MIT Press, 2009. §16.1 (seleção de atividades), §16.2 (mochila fracionária), §16.3 (códigos de Huffman). (Em português: *Algoritmos: Teoria e Prática*, Elsevier, 2012, cap. 16.)
- KLEINBERG, J.; TARDOS, É. *Algorithm Design*. Addison-Wesley, 2006. §4.1, §4.2 e §4.8.
- LEVITIN, A. *Introduction to the Design and Analysis of Algorithms*. 3. ed. Pearson, 2012. §9.4.
- DASGUPTA, S.; PAPADIMITRIOU, C.; VAZIRANI, U. *Algorithms*. McGraw-Hill, 2008. §5.2.
- HUFFMAN, D. A. A Method for the Construction of Minimum-Redundancy Codes. *Proceedings of the IRE*, v. 40, n. 9, p. 1098–1101, 1952.
- COVER, T. M.; THOMAS, J. A. *Elements of Information Theory*. 2. ed. Wiley, 2006. Cap. 5 (entropia e o limite $H \le L < H + 1$).